# 01 · Homografía aplicada a vídeo con suavizado temporal

**Objetivo:** aplicar el pipeline de proyección (detección + keypoints + homografía) a un vídeo completo, generando una pizarra táctica 2D animada sincronizada, con suavizado temporal para evitar el temblor entre frames.

**Contexto:** sobre una imagen la homografía es estable, pero en vídeo las pequeñas variaciones de los keypoints entre frames provocan vibración. Se aplica una media móvil sobre la matriz de homografía y se reutiliza la última matriz válida en frames sin keypoints suficientes.

**Entrada:** vídeo de prueba, modelos `keypoints_base.pt` y `best_v4.pt`.

**Salida:** vídeo de la pizarra 2D animada en `outputs/`.

In [1]:
import numpy as np
import cv2
from ultralytics import YOLO
from pathlib import Path
from collections import deque

REPO_ROOT = Path("../..").resolve()

# Modelos
kp_model = YOLO(str(REPO_ROOT / "models" / "keypoints_base.pt"))
detector = YOLO(str(REPO_ROOT / "models" / "best_v4.pt"))

# Coordenadas reales de los 24 keypoints (metros)
KP_REALES = np.array([
    (0.0,0.0),(0.0,68.0),(105.0,0.0),(105.0,68.0),
    (16.5,13.84),(16.5,54.16),(88.5,13.84),(88.5,54.16),
    (5.5,24.84),(5.5,43.16),(99.5,24.84),(99.5,43.16),
    (11.0,34.0),(94.0,34.0),(52.5,34.0),(52.5,0.0),(52.5,68.0),
    (0.0,13.84),(0.0,54.16),(105.0,13.84),
    (0.0,30.34),(0.0,37.66),(105.0,30.34),(105.0,37.66),
], dtype=np.float32)

EXCLUIR = {14, 15, 16}   # centro, media_inf, media_sup (poco fiables)
CONF_KP = 0.5
CONF_DET = 0.3

print("Modelos cargados")

Modelos cargados


In [2]:
# RUTA DEL VÍDEO DE PRUEBA — ajusta a tu archivo
VIDEO_IN = REPO_ROOT / "data" / "raw" / "A1606b0e6_0 (62).mp4"

print(f"Vídeo existe: {VIDEO_IN.exists()}")
if VIDEO_IN.exists():
    cap = cv2.VideoCapture(str(VIDEO_IN))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f"FPS: {fps}, Frames: {total}, Resolución: {w}x{h}")
    print(f"Duración: {total/fps:.1f} s")

Vídeo existe: True
FPS: 25.0, Frames: 749, Resolución: 1920x1080
Duración: 30.0 s


In [3]:
def calcular_homografia(frame):
    """Detecta keypoints en un frame y devuelve la matriz H (o None si no hay suficientes)."""
    res = kp_model(frame, verbose=False, conf=0.25)
    r = res[0]
    if r.keypoints is None:
        return None
    kps_xy = r.keypoints.xy[0].cpu().numpy()
    kps_conf = r.keypoints.conf[0].cpu().numpy()

    idx = [i for i in range(24) if kps_conf[i] > CONF_KP and i not in EXCLUIR]
    if len(idx) < 4:
        return None

    src = kps_xy[idx]
    dst = KP_REALES[idx]
    H, _ = cv2.findHomography(src, dst, cv2.RANSAC, 3.0)
    return H

def proyectar_jugadores(frame, H):
    """Detecta jugadores y proyecta sus pies al campo. Devuelve lista de (x, y, clase)."""
    res = detector(frame, verbose=False, conf=CONF_DET)
    rd = res[0]
    if len(rd.boxes) == 0:
        return []
    boxes = rd.boxes.xyxy.cpu().numpy()
    clases = rd.boxes.cls.cpu().numpy().astype(int)

    pies = np.array([[[ (x1+x2)/2, y2 ]] for (x1,y1,x2,y2) in boxes], dtype=np.float32)
    proyectados = []
    for p, c in zip(pies, clases):
        pt = cv2.perspectiveTransform(p.reshape(1,1,2), H)[0][0]
        x, y = pt
        # Filtro de cordura: descartar fuera del campo
        if -3 <= x <= 108 and -3 <= y <= 71:
            proyectados.append((x, y, c))
    return proyectados

print("Funciones definidas")

Funciones definidas


## Procesamiento del vídeo con suavizado temporal

Se recorre el vídeo frame a frame. Para cada frame se calcula la homografía; si no hay keypoints suficientes, se reutiliza la última válida. La matriz se suaviza con una media móvil de las últimas N matrices para evitar el temblor. Las posiciones de los jugadores se proyectan y se guardan para la animación.

In [4]:
# Procesar el vídeo y guardar las posiciones proyectadas de cada frame
VENTANA_SUAVIZADO = 5     # nº de matrices para la media móvil
PROCESAR_CADA = 1         # 1 = todos los frames; 2 = uno de cada dos (más rápido)

cap = cv2.VideoCapture(str(VIDEO_IN))
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

historial_H = deque(maxlen=VENTANA_SUAVIZADO)
ultima_H = None
frames_data = []   # lista de listas de (x, y, clase) por frame
H_usadas = 0
H_falladas = 0

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % PROCESAR_CADA == 0:
        H = calcular_homografia(frame)

        if H is not None:
            historial_H.append(H)
            ultima_H = H
            H_usadas += 1
        else:
            H_falladas += 1

        # Matriz suavizada (media de las recientes)
        if len(historial_H) > 0:
            H_suave = np.mean(np.array(historial_H), axis=0)
            jugadores = proyectar_jugadores(frame, H_suave)
        else:
            jugadores = []

        frames_data.append(jugadores)

    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"Procesados {frame_idx}/{total} frames...")

cap.release()
print(f"\nCompletado. Frames con homografía válida: {H_usadas}, fallidos: {H_falladas}")
print(f"Frames de datos guardados: {len(frames_data)}")

Procesados 50/749 frames...
Procesados 100/749 frames...
Procesados 150/749 frames...
Procesados 200/749 frames...
Procesados 250/749 frames...
Procesados 300/749 frames...
Procesados 350/749 frames...
Procesados 400/749 frames...
Procesados 450/749 frames...
Procesados 500/749 frames...
Procesados 550/749 frames...
Procesados 600/749 frames...
Procesados 650/749 frames...
Procesados 700/749 frames...

Completado. Frames con homografía válida: 373, fallidos: 376
Frames de datos guardados: 749


In [6]:
from matplotlib.patches import Circle, Rectangle, Arc

def dibujar_campo(ax):
    """Dibuja un campo de fútbol 2D a escala (105x68 m) sobre el eje dado."""
    ax.add_patch(Rectangle((0, 0), 105, 68, color="#3a7d34", zorder=0))
    lc, lw = "white", 2
    ax.add_patch(Rectangle((0, 0), 105, 68, fill=False, edgecolor=lc, linewidth=lw))
    ax.plot([52.5, 52.5], [0, 68], color=lc, linewidth=lw)
    ax.add_patch(Circle((52.5, 34), 9.15, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((52.5, 34), 0.4, color=lc))
    ax.add_patch(Rectangle((0, 13.84), 16.5, 40.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((0, 24.84), 5.5, 18.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((11, 34), 0.4, color=lc))
    ax.add_patch(Arc((11, 34), 18.3, 18.3, theta1=-53, theta2=53, color=lc, linewidth=lw))
    ax.add_patch(Rectangle((88.5, 13.84), 16.5, 40.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((99.5, 24.84), 5.5, 18.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((94, 34), 0.4, color=lc))
    ax.add_patch(Arc((94, 34), 18.3, 18.3, theta1=127, theta2=233, color=lc, linewidth=lw))
    ax.set_xlim(-5, 110)
    ax.set_ylim(-5, 73)
    ax.set_aspect("equal")
    ax.axis("off")

print("Función dibujar_campo definida")

Función dibujar_campo definida


In [7]:
# Generar el vídeo de la pizarra animada
import matplotlib
matplotlib.use("Agg")  # backend sin ventana, para renderizar a archivo
import matplotlib.pyplot as plt

OUTPUT_VIDEO = REPO_ROOT / "outputs" / "pizarra_animada.mp4"
OUTPUT_VIDEO.parent.mkdir(parents=True, exist_ok=True)

colores = {0: "yellow", 1: "red", 2: "black"}

# Crear el writer de vídeo (tamaño fijo)
PIZARRA_W, PIZARRA_H = 1100, 800
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
fps_salida = fps / PROCESAR_CADA
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps_salida, (PIZARRA_W, PIZARRA_H))

for i, jugadores in enumerate(frames_data):
    fig, ax = plt.subplots(figsize=(11, 8), dpi=100)
    dibujar_campo(ax)

    for (x, y, c) in jugadores:
        ax.scatter(x, y, c=colores.get(c, "blue"), s=120,
                   edgecolors="white", linewidth=1.5, zorder=5)

    # Convertir la figura de matplotlib a imagen para el vídeo
    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    img = buf.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    img = cv2.resize(img, (PIZARRA_W, PIZARRA_H))
    writer.write(img)
    plt.close(fig)

    if i % 50 == 0:
        print(f"Renderizados {i}/{len(frames_data)} frames del vídeo...")

writer.release()
print(f"\nVídeo guardado en: {OUTPUT_VIDEO}")

Renderizados 0/749 frames del vídeo...
Renderizados 50/749 frames del vídeo...
Renderizados 100/749 frames del vídeo...
Renderizados 150/749 frames del vídeo...
Renderizados 200/749 frames del vídeo...
Renderizados 250/749 frames del vídeo...
Renderizados 300/749 frames del vídeo...
Renderizados 350/749 frames del vídeo...
Renderizados 400/749 frames del vídeo...
Renderizados 450/749 frames del vídeo...
Renderizados 500/749 frames del vídeo...
Renderizados 550/749 frames del vídeo...
Renderizados 600/749 frames del vídeo...
Renderizados 650/749 frames del vídeo...
Renderizados 700/749 frames del vídeo...

Vídeo guardado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\outputs\pizarra_animada.mp4
